# Python Teaching Agent + LLM Judge
1. Add `GROQ_API_KEY` in Colab 🔑 Secrets (enable Notebook access).
2. Run the single cell below.


In [ ]:
!pip install -q groq gradio


import json
import os
import re
import subprocess
import sys


def ensure(pkg, module=None):
    """Install a package automatically if it is missing (fixes ModuleNotFoundError)."""
    try:
        __import__(module or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


ensure("groq")
ensure("gradio")

import gradio as gr
from groq import Groq

# ---------------------------------------------------------------
# 0. Setup
# ---------------------------------------------------------------
def get_api_key():
    try:
        from google.colab import userdata  # works inside Colab
        return userdata.get("GROQ_API_KEY")
    except Exception:
        key = os.environ.get("GROQ_API_KEY")
        if not key:
            from getpass import getpass
            key = getpass("Enter your GROQ_API_KEY: ")
        return key


client = Groq(api_key=get_api_key())

# Groq retired llama-3.3-70b-versatile (Aug 2026), so we pick a live model
# automatically: first available one from this preference list.
PREFERRED_MODELS = [
    "openai/gpt-oss-120b",
    "qwen/qwen3.6-27b",
    "openai/gpt-oss-20b",
    "llama-3.3-70b-versatile",
]
SKIP_WORDS = ("whisper", "guard", "tts", "safeguard", "orpheus", "embed")


def pick_model():
    try:
        available = [m.id for m in client.models.list().data]
    except Exception:
        return PREFERRED_MODELS[0]
    for m in PREFERRED_MODELS:
        if m in available:
            return m
    chat_models = [m for m in available if not any(w in m.lower() for w in SKIP_WORDS)]
    return sorted(chat_models)[0] if chat_models else PREFERRED_MODELS[0]


MODEL = pick_model()
print("Using Groq model:", MODEL)


def clean(text):
    """Remove <think>...</think> blocks some reasoning models add."""
    return re.sub(r"<think>.*?</think>", "", text or "", flags=re.S).strip()

# ---------------------------------------------------------------
# 1. Python Teaching Agent  (system prompt)
# ---------------------------------------------------------------
TEACHER_SYSTEM_PROMPT = """
You are "PyBuddy", a friendly, patient and encouraging Python programming tutor
for absolute beginners.

ROLE
- You teach basic Python: variables, data types, operators, conditional
  statements (if/elif/else), loops (for/while), functions, lists and strings.
- If a question is not about Python or programming, politely steer the student
  back to learning Python.

HOW YOU EXPLAIN
- Use simple, everyday language. Avoid jargon; if you must use a technical
  term, explain it in one short sentence.
- Start with a real-life analogy (e.g. "a variable is like a labelled box").
- Keep explanations short: small paragraphs, bullet points where helpful.

HOW YOU GIVE EXAMPLES
- Always include at least one small, runnable code snippet in a ```python block.
- Add the expected output of the code (as a comment or an "Output:" block).
- Walk through the code line by line in plain words.

HOW YOU GUIDE (DON'T JUST GIVE ANSWERS)
- When a student asks you to solve a homework-style problem, do NOT hand over
  the full solution. Break it into steps, give hints, show a similar example,
  and ask guiding questions so the student reaches the answer themselves.
- If they are stuck after trying, give a bigger hint, not the full answer.
- End every reply with ONE short practice question or mini-challenge.

ADAPTING TO THE STUDENT
- Assume the student is a complete beginner unless they show otherwise.
- If they seem confused, re-explain with a simpler analogy and smaller steps.
- If they show more experience, add one slightly deeper tip.
- Be encouraging and positive; mistakes are part of learning.
""".strip()


def teach(history, user_message):
    """Call the teaching agent with the running conversation."""
    messages = [{"role": "system", "content": TEACHER_SYSTEM_PROMPT}]
    for turn in history:
        messages.append({"role": turn["role"], "content": turn["content"]})
    messages.append({"role": "user", "content": user_message})

    resp = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.6,
        max_tokens=2500,  # reasoning models spend some tokens thinking
    )
    return clean(resp.choices[0].message.content)


# ---------------------------------------------------------------
# 2. LLM Judge (Teaching Evaluator)
# ---------------------------------------------------------------
CRITERIA = [
    "conceptual_accuracy",
    "clarity",
    "beginner_friendliness",
    "example_quality",
    "teaching_approach",
]

JUDGE_SYSTEM_PROMPT = """
You are a strict but fair Teaching Evaluator (LLM Judge). You evaluate how well
an AI Python tutor answered a beginner's question.

RUBRIC (score each criterion with an INTEGER from 0 to 5):
1. conceptual_accuracy      - Is the Python content technically correct?
2. clarity                  - Is the explanation clear, well organised, easy to follow?
3. beginner_friendliness    - Simple language, analogies, no unexplained jargon?
4. example_quality          - Are code examples relevant, correct, runnable, explained?
5. teaching_approach        - Does it guide the learner (hints, questions, practice)
                              rather than just dumping answers?

Scoring guide: 0 = missing/wrong, 1 = poor, 2 = weak, 3 = acceptable,
4 = good, 5 = excellent. Do not give all 5s by default.

Return ONLY a valid JSON object with exactly these keys:
{
  "conceptual_accuracy": <int 0-5>,
  "clarity": <int 0-5>,
  "beginner_friendliness": <int 0-5>,
  "example_quality": <int 0-5>,
  "teaching_approach": <int 0-5>,
  "total_score": <int, sum of the five scores, out of 25>,
  "feedback": "<2-3 sentences explaining the evaluation>",
  "suggestions": "<specific suggestions to improve the response>"
}
No markdown, no extra text.
""".strip()


def judge(question, answer):
    """Second LLM call: evaluate the teaching agent's answer."""
    user_prompt = (
        f"STUDENT QUESTION:\n{question}\n\n"
        f"TUTOR RESPONSE:\n{answer}\n\n"
        "Evaluate the tutor response using the rubric."
    )
    messages = [
        {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    try:
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                temperature=0,
                max_tokens=2500,
                response_format={"type": "json_object"},  # forces structured JSON
            )
        except Exception:  # model without JSON mode -> retry without it
            resp = client.chat.completions.create(
                model=MODEL, messages=messages, temperature=0, max_tokens=2500
            )
        raw = clean(resp.choices[0].message.content)
        match = re.search(r"\{.*\}", raw, flags=re.S)  # grab the JSON object
        data = json.loads(match.group(0) if match else raw)
    except Exception as e:
        return {
            **{c: 0 for c in CRITERIA},
            "total_score": 0,
            "feedback": f"Judge failed to produce a valid evaluation: {e}",
            "suggestions": "Try again.",
        }

    # Validate / clamp scores and recompute the total ourselves
    result = {}
    for c in CRITERIA:
        try:
            result[c] = max(0, min(5, int(data.get(c, 0))))
        except (TypeError, ValueError):
            result[c] = 0
    result["total_score"] = sum(result[c] for c in CRITERIA)
    result["feedback"] = str(data.get("feedback", ""))
    result["suggestions"] = str(data.get("suggestions", ""))
    return result


# ---------------------------------------------------------------
# 3. UI helpers
# ---------------------------------------------------------------
LABELS = {
    "conceptual_accuracy": "Conceptual Accuracy",
    "clarity": "Clarity of Explanation",
    "beginner_friendliness": "Beginner Friendliness",
    "example_quality": "Quality of Examples",
    "teaching_approach": "Teaching Approach",
}


def bar_color(score):
    if score >= 4:
        return "#22a559"
    if score == 3:
        return "#e0a800"
    return "#d64545"


def render_evaluation(ev):
    total = ev["total_score"]
    total_color = bar_color(round(total / 5))
    rows = ""
    for c in CRITERIA:
        s = ev[c]
        rows += f"""
        <div style="margin:8px 0;">
          <div style="display:flex;justify-content:space-between;font-size:14px;">
            <span>{LABELS[c]}</span><b>{s} / 5</b>
          </div>
          <div style="background:#e5e7eb;border-radius:6px;height:10px;">
            <div style="width:{s * 20}%;background:{bar_color(s)};height:10px;border-radius:6px;"></div>
          </div>
        </div>"""
    return f"""
    <div style="font-family:sans-serif;padding:6px 2px;">
      <div style="text-align:center;margin-bottom:10px;">
        <div style="font-size:13px;opacity:.7;">TOTAL SCORE</div>
        <div style="font-size:42px;font-weight:700;color:{total_color};">{total}<span style="font-size:20px;opacity:.6;"> / 25</span></div>
      </div>
      {rows}
      <div style="margin-top:14px;font-size:14px;"><b>Feedback:</b> {ev['feedback']}</div>
      <div style="margin-top:8px;font-size:14px;"><b>Suggestions:</b> {ev['suggestions']}</div>
    </div>"""


EMPTY_EVAL = "<div style='padding:20px;opacity:.6;'>Ask a question to see the judge's evaluation here.</div>"


def respond(user_message, history):
    if not user_message.strip():
        return history, "", EMPTY_EVAL, None
    history = history or []

    answer = teach(history, user_message)          # Step 1: teaching agent
    evaluation = judge(user_message, answer)       # Step 2: LLM judge

    history = history + [
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": answer},
    ]
    return history, "", render_evaluation(evaluation), evaluation


# ---------------------------------------------------------------
# 4. Gradio UI
# ---------------------------------------------------------------
with gr.Blocks(title="Python Teaching Agent + LLM Judge") as demo:
    gr.Markdown(
        "# 🐍 Python Teaching Agent with LLM Judge\n"
        "Ask a beginner Python question. The **tutor** answers, then a second LLM "
        "(the **judge**) scores the answer on a 5-point rubric."
    )
    with gr.Row():
        with gr.Column(scale=3):
            try:
                chatbot = gr.Chatbot(label="PyBuddy (Teaching Agent)", height=520, type="messages")
            except TypeError:  # newer Gradio versions use messages by default
                chatbot = gr.Chatbot(label="PyBuddy (Teaching Agent)", height=520)
            msg = gr.Textbox(
                placeholder="e.g. What is a Python variable?",
                label="Your question",
                lines=1,
            )
            with gr.Row():
                send = gr.Button("Ask", variant="primary")
                clear = gr.Button("Clear")
            gr.Examples(
                examples=[
                    "What is a Python variable?",
                    "How does a for loop work?",
                    "What is the difference between a list and a string?",
                    "Explain functions to me like I'm 12.",
                ],
                inputs=msg,
            )
        with gr.Column(scale=2):
            gr.Markdown("### ⚖️ LLM Judge Evaluation")
            eval_html = gr.HTML(EMPTY_EVAL)
            with gr.Accordion("Raw JSON output", open=True):
                eval_json = gr.JSON(label="Structured evaluation")

    outputs = [chatbot, msg, eval_html, eval_json]
    send.click(respond, [msg, chatbot], outputs)
    msg.submit(respond, [msg, chatbot], outputs)
    clear.click(lambda: ([], "", EMPTY_EVAL, None), None, outputs)

try:  # Gradio 6: theme is passed to launch()
    demo.launch(theme=gr.themes.Soft(), share=True, debug=True)
except TypeError:  # Gradio 4/5
    demo.launch(share=True, debug=True)
